In [ ]:
pip install librosa resemblyzer pymcd soundfile numpy scipy

In [ ]:
import os
import numpy as np
import librosa

try:
    from resemblyzer import VoiceEncoder, preprocess_wav
    from pymcd.mcd import Calculate_MCD
except ImportError:
    print("⚠️ Missing libraries! Please run:")
    print("pip install librosa resemblyzer pymcd")
    exit()

# ==========================================
# 1. DEFINE YOUR AUDIO FILES
# ==========================================
# Point these to wherever your audio files are stored
REFERENCE_WAV = "target_reference.wav"  
OUTPUT_FILENAME = "generated_output.wav"

if not os.path.exists(REFERENCE_WAV) or not os.path.exists(OUTPUT_FILENAME):
    raise FileNotFoundError("⚠️ Audio files not found! Please check your file paths.")

# ==========================================
# 2. EVALUATION PIPELINE
# ==========================================
print("\n" + "="*50)
print("📊 FINAL RESEARCH METRICS REPORT")
print("="*50)

def get_prosody_stats(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = f0[~np.isnan(f0)]
    rms = librosa.feature.rms(y=y)[0]
    return {
        "mean_f0": np.mean(f0) if len(f0) > 0 else 0,
        "std_f0": np.std(f0) if len(f0) > 0 else 0,
        "mean_energy": np.mean(rms)
    }

target_stats = get_prosody_stats(REFERENCE_WAV)
bridge_stats = get_prosody_stats(OUTPUT_FILENAME)

print("1️⃣ PROSODY & EMPATHY ALIGNMENT")
print("-" * 45)
print(f"{'Metric':<18} | {'Target':<12} | {'Bridge Out':<12}")
print("-" * 45)
print(f"{'Mean Pitch (Hz)':<18} | {target_stats['mean_f0']:>12.2f} | {bridge_stats['mean_f0']:>12.2f}")
print(f"{'Pitch StdDev':<18} | {target_stats['std_f0']:>12.2f} | {bridge_stats['std_f0']:>12.2f}")
print(f"{'Mean Energy':<18} | {target_stats['mean_energy']:>12.6f} | {bridge_stats['mean_energy']:>12.6f}")
print("-" * 45)

# 2️⃣ ACOUSTIC DISTORTION
mcd_toolbox = Calculate_MCD(MCD_mode="dtw")
mcd_score = mcd_toolbox.calculate_mcd(REFERENCE_WAV, OUTPUT_FILENAME)
print(f"\n2️⃣ ACOUSTIC DISTORTION")
print("-" * 45)
print(f"📉 Official DTW-MCD Score:      {mcd_score:.2f} dB")

# 3️⃣ BIOMETRIC IDENTITY TRANSFER
print(f"\n3️⃣ BIOMETRIC IDENTITY TRANSFER")
print("-" * 45)
try:
    # Initialize the VoiceEncoder
    encoder = VoiceEncoder()
    
    # Preprocess the audio files (Resemblyzer applies VAD and normalization)
    wav_target_processed = preprocess_wav(REFERENCE_WAV)
    wav_output_processed = preprocess_wav(OUTPUT_FILENAME)
    
    # Generate 256-dimensional d-vectors
    emb_target = encoder.embed_utterance(wav_target_processed)
    emb_generated = encoder.embed_utterance(wav_output_processed)
    
    # Calculate Cosine Similarity via inner product (vectors are L2 normalized)
    similarity = np.inner(emb_target, emb_generated)
    
    print(f"🧬 Identity Similarity Score: {similarity:.4f}")
    
    if similarity > 0.75:
        print("🟢 RESULT: Strong Identity Match!")
    elif similarity > 0.60:
        print("🟡 RESULT: Moderate Identity Match (Perceptually similar)")
    else:
        print("🔴 RESULT: Weak Match")
        
except Exception as e:
    print(f"⚠️ Identity Check Failed: {e}")

print(f"\n{'='*75}\n⚠️ RESEARCHER NOTE: Kaggle API silent hardware downgrades (Dual-T4 to P100) caused 16-bit OOM errors. So, running Full Inference with Kaggle API on Opencode CLI was not possible. Metrics use pre-generated audio produced by our Soundstorm model for evaluation.\n{'='*75}\n")

⚠️ Missing resemblyzer! Run: !pip install resemblyzer


ModuleNotFoundError: No module named 'moshi'